# LwF: Evaluación Class-IL

Este notebook implementa **Learning without Forgetting** en el escenario **Class-IL**.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import ConcatDataset, DataLoader
from models import CNN, ClassIncrementalClassifier
from dataloaders import SequentialCIFAR10
from utils_class_il import evaluate_class_il
from copy import deepcopy

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
BATCH_SIZE = 128
TEMPERATURE = 2.0
ALPHA = 1.0
EPOCHS = 10
print(f"Usando dispositivo: {device}")

Usando dispositivo: mps


In [2]:
def distillation_loss(logits, targets, temp):
    log_p = F.log_softmax(logits / temp, dim=1)
    p = F.softmax(targets / temp, dim=1)
    return - (p * log_p).sum(dim=1).mean() * (temp**2)

In [3]:
seq_cifar = SequentialCIFAR10(batch_size=BATCH_SIZE)
backbone = CNN(in_channels=3, embedding_dim=32)
model = ClassIncrementalClassifier(backbone, embedding_dim=32, total_classes=10).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

prev_model = None

for task_id in range(5):
    print(f"\n--- ENTRENANDO TAREA {task_id} ---")
    model.add_task(seq_cifar.task_classes[task_id])
    train_ds = seq_cifar.get_task_train_dataset(task_id, remap_labels=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            
            # Pérdida de clasificación (Tarea actual)
            loss_ce = nn.CrossEntropyLoss()(logits, y)
            
            # Pérdida de destilación (Tareas pasadas)
            loss_kd = 0
            if prev_model is not None:
                with torch.no_grad():
                    old_logits = prev_model(x)
                # Solo destilamos las clases de tareas anteriores
                prev_classes_mask = prev_model.active_classes == 1
                loss_kd = distillation_loss(logits[:, prev_classes_mask], old_logits[:, prev_classes_mask], TEMPERATURE)
            
            loss = loss_ce + ALPHA * loss_kd
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"  Epoch {epoch+1}/{EPOCHS} | Loss: {running_loss/len(train_loader):.4f}")
            
    # Guardar modelo para la siguiente tarea
    prev_model = deepcopy(model)
    prev_model.eval()
    
    # Evaluar Class-IL
    all_test = ConcatDataset([seq_cifar.get_task_test_dataset(tid) for tid in range(task_id+1)])
    acc = evaluate_class_il(model, DataLoader(all_test, batch_size=BATCH_SIZE), device, [])
    print(f"Precisión Class-IL tras Tarea {task_id}: {acc:.2f}%")


--- ENTRENANDO TAREA 0 ---
  Epoch 1/10 | Loss: 0.4845
  Epoch 2/10 | Loss: 0.3640
  Epoch 3/10 | Loss: 0.3233
  Epoch 4/10 | Loss: 0.2850
  Epoch 5/10 | Loss: 0.2673
  Epoch 6/10 | Loss: 0.2668
  Epoch 7/10 | Loss: 0.2442
  Epoch 8/10 | Loss: 0.2420
  Epoch 9/10 | Loss: 0.2240
  Epoch 10/10 | Loss: 0.2326
Precisión Class-IL tras Tarea 0: 92.75%

--- ENTRENANDO TAREA 1 ---
  Epoch 1/10 | Loss: 2.9303
  Epoch 2/10 | Loss: 2.7896
  Epoch 3/10 | Loss: 2.7641
  Epoch 4/10 | Loss: 2.7350
  Epoch 5/10 | Loss: 2.7512
  Epoch 6/10 | Loss: 2.7406
  Epoch 7/10 | Loss: 2.7317
  Epoch 8/10 | Loss: 2.7260
  Epoch 9/10 | Loss: 2.7192
  Epoch 10/10 | Loss: 2.7012
Precisión Class-IL tras Tarea 1: 38.90%

--- ENTRENANDO TAREA 2 ---
  Epoch 1/10 | Loss: 3.7998
  Epoch 2/10 | Loss: 3.2132
  Epoch 3/10 | Loss: 3.1666
  Epoch 4/10 | Loss: 3.1328
  Epoch 5/10 | Loss: 3.1296
  Epoch 6/10 | Loss: 3.1057
  Epoch 7/10 | Loss: 3.1021
  Epoch 8/10 | Loss: 3.0900
  Epoch 9/10 | Loss: 3.0920
  Epoch 10/10 | Loss: 